<div align=center><font size='5'>《机器学习与 Python 编程》研究性专题</font></div>

<div align=center><font size='5'>——裂纹图像识别智能平台</font></div>


## 一、研究背景
在工业领域，裂纹检测是一项至关重要的工作。裂纹的存在可能导致材料疲劳破坏，影响设备的安全性和性能。因此，及时准确地检测裂纹对于预测材料的寿命、减少设备损坏以及提高生产效率具有重要意义。

传统的裂纹检测方法通常依赖于人工观察和经验判断，这种方式存在一定的主观性和效率低下的问题。人工裂纹检测需要经验丰富的技术人员进行，且耗时费力。由于这种情况，许多研究人员和工程师开始寻求更高效、更准确的裂纹检测方法。利用计算机视觉和机器学习技术进行裂纹图像的自动检测和分类成为一个备受关注的研究方向。计算机视觉技术可以帮助自动识别裂纹，减少了对于复杂图像的主观判断。同时，结合机器学习算法，可以让系统通过学习裂纹的特征，不断提高准确性和效率。这种技术的引入可以提高裂纹检测的自动化程度，减少人为干预，从而提高工作效率，并降低检测错误率。

因此，利用计算机视觉和机器学习技术进行裂纹图像的自动检测和分类，在裂纹检测领域具有巨大的潜在价值，可以为工业生产带来更高的效益和性能保障。

## 二、数据集划分
1.留出法
    
留出法是最简单的数据划分方法之一。该方法从原始数据集中随机选取一部分数据作为训练集，剩余部分作为测试集。通常情况下，训练集占总数据集的大部分比例，而测试集则占较小的比例。留出法简单直观，易于理解和实现。可以有效评估模型在未见过的数据上的泛化能力。但由于仅进行一次数据划分，模型性能评估可能对数据的选择敏感，存在一定的随机性。

2.5折交叉验证法：

数据划分方式：5折交叉验证将原始数据集平均分为5份，其中一份作为验证集，其余四份作为训练集。经过5轮训练和验证，每一份数据都会作为验证集一次，从而获得五次性能评估结果的平均值。它能够更准确地评估模型的泛化能力，减少单次数据划分带来的不确定性。能够充分利用数据，避免过拟合或欠拟合。但是计算开销相对较大，因为需要多次训练模型。对于较大规模的数据集和复杂的模型，可能会增加训练时间。

3.方案设计及代码

+ 留出法数据集划分设计步骤：

  (1) 导入必要的库；

  (2) 定义辅助函数；

  (3) 设置随机种子；

  (4) 定义数据集路径和保存路径；

  (5) 定义数据集划分比例；

  (6) 数据划分；

  (7) 输出每个类别的数据分布。

+ 5折交叉验证法数据集划分设计步骤：

  (1) 导入必要的库；

  (2) 定义辅助函数；

  (3) 设置随机种子；

  (4) 定义数据集路径和保存路径；

  (5) 定义KFold对象；

  (6) 数据划分；

  (7) 遍历每个fold；

  (8) 输出每个fold的数据分布。


导入必要的库

In [1]:
import os
from sklearn import svm
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import hinge_loss
from functools import partial
from sklearn.svm import LinearSVC

留出法进行数据集划分

In [2]:
#留出法
# 创建保存图像的文件夹
def makedir(new_dir):
    if not os.path.exists(new_dir):
        os.makedirs(new_dir)
random.seed(1) # 随机种子

# 1.确定原图像数据集路径
dataset_dir = "E:\学习\python\Dataset"  #原始数据集路径
# 2.确定数据集划分后保存的路径
split_dir = "E:\学习\python\Dataset1"  #划分后保存路径
train_dir = os.path.join(split_dir, "train")
valid_dir = os.path.join(split_dir, "val")
test_dir = os.path.join(split_dir, "test")
# 3.确定将数据集划分为训练集，验证集，测试集的比例
train_pct = 0.8
valid_pct = 0.1
test_pct = 0.1
# 4.划分
# 遍历数据集文件夹
for root, dirs, files in os.walk(dataset_dir):
    # 遍历每个类别的文件夹
    for sub_dir in dirs:
        # 获取当前类别文件夹下的所有图像文件
        imgs = os.listdir(os.path.join(root, sub_dir))
        # 筛选出以'.jpg'结尾的图像文件
        imgs = list(filter(lambda x: x.endswith('.jpg'), imgs))
        # 打乱图像文件顺序
        random.shuffle(imgs)
        # 计算该类别下图像的数量
        img_count = len(imgs)
        # 计算训练集、验证集和测试集的划分点
        train_point = int(img_count * train_pct)  # 训练集的划分点
        valid_point = int(img_count * (train_pct + valid_pct))  # 验证集的划分点

        # 遍历该类别下的每张图像
        for i in range(img_count):
            # 根据不同的划分点确定图像的目标保存文件夹
            if i < train_point:  # 保存到训练集
                out_dir = os.path.join(train_dir, sub_dir)
            elif i < valid_point:  # 保存到验证集
                out_dir = os.path.join(valid_dir, sub_dir)
            else:  # 保存到测试集
                out_dir = os.path.join(test_dir, sub_dir)
            # 创建目标保存文件夹
            os.makedirs(out_dir, exist_ok=True)
            # 构建图像的目标保存路径和源路径
            target_path = os.path.join(out_dir, imgs[i])
            src_path = os.path.join(dataset_dir, sub_dir, imgs[i])
            # 复制图像文件到目标保存路径
            shutil.copy(src_path, target_path)

        print('Class:{}, train:{}, valid:{}, test:{}'.format(sub_dir, train_point, valid_point-train_point, img_count-valid_point))


<>:9: SyntaxWarning: invalid escape sequence '\p'
<>:11: SyntaxWarning: invalid escape sequence '\p'
<>:9: SyntaxWarning: invalid escape sequence '\p'
<>:11: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_638/2884681990.py:9: SyntaxWarning: invalid escape sequence '\p'
  dataset_dir = "E:\学习\python\Dataset"  #原始数据集路径
/tmp/ipykernel_638/2884681990.py:11: SyntaxWarning: invalid escape sequence '\p'
  split_dir = "E:\学习\python\Dataset1"  #划分后保存路径


NameError: name 'random' is not defined

5折交叉验证法进行数据划分

In [ ]:
    #s折交叉验证法
for root, dirs, files in os.walk(dataset_dir):
    path_targe = "E:\学习\python\Dataset2" #保存划分后的数据集地址

    # 判断是否存在并重新创建文件夹
    if os.path.exists(path_targe):
        shutil.rmtree(path_targe)
        os.makedirs(path_targe)
        print("succeed : ", path_targe)
    else:
        os.makedirs(path_targe)
        print("succeed : ", path_targe)


if __name__ == '__main__':
    # ==================设置超参数===================== #
    #  root_data : 数据路径    格式:文件名->n个类别->m张图像(需符合pytorch读取训练数据集要求)
    root_data = "EE:\学习\python\Dataset"  #划分前的数据集地址
    #  save_path:存放数据路径
    save_path = "E:\学习\python\Dataset2"
    #  k_num : 设置交叉验证数
    k_num = 5
    # =============================================== #
    # 定义交叉验证:设置参数  shuffle=True 打乱, random_state随机数种子
    kf = KFold(n_splits=k_num, shuffle=True, random_state=0)

    # 生成存放文件目录,这部分代码遍历了数据集中的每个类别文件夹，并对每个类别进行处理。对于每个类别，它首先检查是否是一个目录，然后获取该类别下的所有图片文件列表 file_sub。
   #接下来，通过 kf.split(file_sub) 来生成交叉验证的训练集和测试集索引。
    save_path = os.path.join(save_path)
    file = os.listdir(root_data)
    for i in file:
        root_sub = os.path.join(root_data, i)
        flag = os.path.isdir(root_sub)  # 验证root_sub是否为目录
        if flag:
            file_sub = os.listdir(root_sub)
            print(file_sub)
            for n, data in enumerate(kf.split(file_sub)):
                train, test = data
                save_path_sub_train = os.path.join(save_path, str(n), "train", i)
                save_path_sub_test = os.path.join(save_path, str(n), "test", i)
                # 生成路径
                os.makedirs(save_path_sub_train, exist_ok=True)
                os.makedirs(save_path_sub_test, exist_ok=True)
                print(len(train), len(test))
                for item in train:
                    img_name = file_sub[item]
                    # 图像路径
                    path_train = os.path.join(root_sub, img_name)
                    # 目标路径
                    targe_path = os.path.join(save_path_sub_train, img_name)
                    # 开始移动
                    shutil.copy(path_train, targe_path)
                for item in test:
                    img_name = file_sub[item]
                    # 图像路径
                    path_train = os.path.join(root_sub, img_name)
                    # 目标路径
                    targe_path = os.path.join(save_path_sub_test, img_name)
                    # 开始移动
                    shutil.copy(path_train, targe_path)
    print("End...{}折交叉验证已完成 ,  数据集路径为: {}".format(k_num, save_path))


4.性能比较与分析

留出法：

  优势：

  留出法简单直观，易于理解和实施。

  对于大型数据集而言，留出法可能提供了足够的样本用于训练和测试，从而能够提供合理的分类性能评估。

  劣势：

  由于留出法将一部分数据保留为测试集，因此训练数据相对较少，可能导致模型的性能评估不够准确。

  结果可能依赖于数据的初始划分方式，从而可能不够稳健。

S折交叉验证：

  优势：

  S折交叉验证充分利用了数据集中的所有样本进行训练和验证，因此可以更准确地评估模型的分类性能。

  通过多次训练和验证的平均结果，可以降低由于特定数据划分引起的偏差，提高了结果的稳健性。

  劣势：

  计算成本较高，因为需要进行多次模型训练和评估。

  对于大型数据集和复杂模型，S折交叉验证可能需要较长的训练时间。

  需要在多个训练和验证周期中进行结果的聚合和分析，相对复杂。

总体比较：

  如果数据集足够大，并且计算资源有限，留出法是一个更合适的选择，因为它的计算成本较低。
  
  如果对模型的性能评估要求较高，或者数据集较小，S折交叉验证可能是更可靠的选择，因为它提供了更准确和稳健的性能评估。




## 三、模型选择
1.卷积神经网络CNN

卷积神经网络（CNN）是一类包含卷积计算且具有深度结构的前馈神经网络，是深度学习的代表算法之一。CNN具有表征学习能力，能够按其层次结构对输入信息进行平移不变分类，因此也被称为“平移不变人工神经网络”。对CNN的研究始于20世纪80至90年代，LeNet-5是最早出现的CNN之一。随着深度学习理论的提出和计算设备的改进，CNN得到了快速发展，并被应用于计算机视觉、自然语言处理等领域。其多层网络结构能自动学习输入数据的深层特征，不同层次的网络可以学习到不同层次的特征。
  
卷积神经网络CNN模型设计思路：

（1）读取图像函数(read_image)：这个函数接收一个文件路径作为输入，使用OpenCV库(cv2)读取图像，并进行了两个操作：颜色通道转换和大小调整。颜色通道转换将图像的颜色通道从BGR转换为RGB，而大小调整将图像的大小调整为28x28像素。最后返回的图像是一个浮点数类型的Numpy数组。

（2）TensorFlow模型构建：使用TensorFlow的Keras API构建一个顺序模型。该模型包含了一系列层，包括两个卷积层(Conv2D)、两个最大池化层(MaxPooling2D)、一个展平层(Flatten)、两个全连接层(Dense)和一个Dropout层。这些层的配置是根据卷积神经网络的典型结构来设计的，用于图像分类任务。

（3）模型训练：使用model.fit函数对模型进行训练。传入训练集(X_train, y_train)，设置了训练的轮数(epochs)为10，批量大小(batch_size)为32。

（4）模型编译：使用model.compile函数编译模型。在这里，选择了Adam优化器(optimizer)，二元交叉熵(binary_crossentropy)作为损失函数(loss)，并指定了准确率(accuracy)作为评估指标(metrics)。

（5）模型评估：使用model.evaluate函数评估模型在测试集上的性能。评估结果包括测试集上的损失值(test_loss)和准确率(test_acc)。

（6）保存模型：使用model.save函数将训练好的模型保存到名为my_model.keras的文件中，以便之后可以加载和使用。


卷积神经网络CNN模型设计程序：

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
import glob

# TensorFlow模型构建
model = tf.keras.models.Sequential([  #创建了一个序贯模型，即一系列层的线性堆叠。
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 3)),#卷积层，
    tf.keras.layers.MaxPooling2D((2, 2)),#最大池化层，用于提取图像特征并减小维度。
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),#展平层 ，扁平化处理，将卷积层的输出展平成一维向量。
    tf.keras.layers.Dense(1024, activation='relu'), #全连接层 (tf.keras.layers.Dense)，包含 1024 个神经元，并使用 ReLU 激活函数
    tf.keras.layers.Dropout(0.5),   #减少过拟合，0.5：表示 Dropout 操作中随机丢弃的神经元比例，这里是 50% 的神经元被丢弃。
    tf.keras.layers.Dense(1, activation='sigmoid')    #1：输出层中神经元的数量，因为是二分类任务，所以只有一个神经元。
#激活函数在神经网络中起着非常重要的作用，它们通常被应用于每个神经元的输出，用于引入非线性特性和增加网络的表达能力。
    #ReLU（Rectified Linear Unit）是一种常用的激活函数之一，即输入小于零时输出为零，输入大于零时输出为输入值本身。
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
#使用 model.compile 编译模型，
#指定了优化器为 Adam (optimizer='adam')，损失函数为二元交叉熵 (loss='binary_crossentropy')，评估指标为准确率 (metrics=['accuracy'])。
# 模型训练
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# 模型评估
test_loss, test_acc = model.evaluate(X_test, y_test)
print('Test accuracy:', test_acc)

# 保存模型
model.save('my_model.keras')


2.残差网络ResNet：

残差网络（ResNet）是一种深度神经网络架构，提出了残差学习的概念，通过引入残差连接解决了深度网络训练中的梯度消失和梯度爆炸问题。ResNet由多个残差块组成，每个残差块包含两个或三个卷积层。残差连接允许前一层的激活作为输入添加到后一层的激活中，使得网络可以学习残差映射。ResNet在图像分类、物体检测、图像分割等任务中取得了巨大成功，提高了网络的训练效率和准确性。

残差网络ResNet模型设计思路：

（1）定义一个名为 CracksDataset 的数据集类。在初始化时，它接受一个名为 data_folders 的参数，这是包含图像数据的文件夹路径列表，以及一个可选的 transform 参数，用于数据预处理和增强。

（2）创建 CracksDataset 类的一个实例 cracks_dataset，并传入 data_folders 和 transform 作为参数，从而加载和预处理数据集中的图像数据。

（3）加载预训练的 ResNet-18 模型。这个模型是在 ImageNet 数据集上预训练的，weights 参数指定了使用的预训练权重。

（4）将 ResNet 模型的最后一层全连接层替换为一个新的线性层，输出维度为2,用于进行二分类任务。

（5）对于每个批次中的图片和标签，执行以下操作：

    ①清零优化器的梯度，以避免梯度累积。

    ②将图片输入模型，获取模型的输出。

    ③计算模型输出与真实标签之间的损失。

    ④反向传播，计算损失相对于模型参数的梯度。
    
    ⑤根据梯度更新模型参数，执行一步优化。




残差网络ResNet模型设计程序：

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os

# 加载裂痕数据集
data_folders = ['E:/pythonlearning/kfold/4/train']
cracks_dataset = CracksDataset(data_folders, transform=transform)

# 创建数据加载器
batch_size = 32  #每次输入32张图片进行训练。
data_loader = DataLoader(cracks_dataset, batch_size=batch_size, shuffle=True)#创建了一个数据加载器对象 data_loader，用于加载训练数据集

# 加载预训练的ResNet模型
resnet_model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)

# 修改模型的最后一层，使其输出维度为2（二分类）
num_ftrs = resnet_model.fc.in_features
resnet_model.fc = torch.nn.Linear(num_ftrs, 2)   #将 ResNet 模型的最后一层全连接层替换为一个新的线性层，输出维度为2,用于进行二分类任务。

# 将模型切换到训练模式
resnet_model.train()

# 定义损失函数和优化器
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(resnet_model.parameters(), lr=0.001)

# 训练模型
num_epochs = 10
for epoch in range(num_epochs):
    for images, labels in data_loader:
        optimizer.zero_grad()  #清零优化器的梯度，以避免梯度累积。
        outputs = resnet_model(images)  #将图片输入模型，获取模型的输出。
        loss = criterion(outputs, labels)   #计算模型输出与真实标签之间的损失。
        loss.backward()      #计算模型输出与真实标签之间的损失，反向传播计算梯度
        optimizer.step()    #根据梯度更新模型参数，执行一步优化。
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")
# 保存训练好的模型
torch.save(resnet_model.state_dict(), 'crack_detection_model.pth')






3.支持向量机SVM：

支持向量机（SVM）是一种监督学习算法，用于分类和回归任务。SVM通过找到一个最优的超平面来将不同类别的数据分开，使得两类数据点之间的间隔最大化。SVM具有良好的泛化能力和对小样本数据的处理能力，在文本分类、图像识别、生物信息学等领域得到了广泛应用。

支持向量机SVM模型设计思路：

（1）获取图像数据集和标签集。导入数据集的路径保存为数组，遍历路径数组，将图像转化为灰度图，调整大小为64*64，并展开为一维数组。（为了计算和存储方便）同时判断文件路径的Negative和Positive，获取图像标签。最后得图像数据集和标签集。

（2）处理数据。进行PCA降维处理，保留80%的主成分，返回降维后的数据。

（3）五折交叉验证训练SVM模型。随机将数据集切分为5个互不相交、大小相同的子集；利用4个子集的数据训练SVM模型，剩余的一个子集用于测试；利用可能的5种选择进行评价，平均5次评价的结果作为最终的评价结果。保存SVM模型，获取最优模型。





支持向量机SVM模型设计程序：

In [ ]:

train_dir = "E:\\学习\\python\\keshe\\dataset2\\train"
val_dir = "E:\\学习\\python\\keshe\\dataset2\\val"
test_dir = "E:\\学习\\python\\keshe\\dataset2\\test"

def extract_features(image_path):
    # 从图像路径中提取图片是否有裂分特征
    category = os.path.basename(os.path.dirname(image_path))  # 提取文件夹名称（Positive 或 Negative）

    # 根据类别确定特征值：有裂分为 1（Positive），无裂分为 0（Negative）
    has_crack = 1 if category == 'Positive' else 0

    # 构建特征字典
    feature_dict = {'has_crack': has_crack}

    return feature_dict

#加载图片
def load_dataset(data_dir):
    features = []
    labels = []

    for class_name in os.listdir(data_dir):
        class_dir = os.path.join(data_dir, class_name)
        if os.path.isdir(class_dir):
            for image_file in os.listdir(class_dir):
                if image_file.endswith('.jpg'):
                    image_path = os.path.join(class_dir, image_file)
                    features.append(extract_features(image_path))
                    labels.append(class_name)

    return features, labels


# 加载训练集数据
train_features, train_labels = load_dataset(train_dir)

# 加载验证集数据
val_features, val_labels = load_dataset(val_dir)

# 加载测试集数据
test_features, test_labels = load_dataset(test_dir)

# 编码标签
label_encoder = LabelEncoder()
train_labels_encoded = label_encoder.fit_transform(train_labels)
val_labels_encoded = label_encoder.transform(val_labels)
test_labels_encoded = label_encoder.transform(test_labels)

# 将特征字典转换为向量
vectorizer = DictVectorizer(sparse=False)
train_features_encoded = vectorizer.fit_transform(train_features)
val_features_encoded = vectorizer.transform(val_features)
test_features_encoded = vectorizer.transform(test_features)

# 训练SVM模型
svm_classifier = svm.SVC(kernel='linear', C=1.0)
svm_classifier.fit(train_features_encoded, train_labels_encoded)

# 在验证集上进行预测
val_predictions = svm_classifier.predict(val_features_encoded)
accuracy_val = accuracy_score(val_labels_encoded, val_predictions)
print("Hinge验证集准确率:", accuracy_val)

# 在测试集上进行预测
test_predictions = svm_classifier.predict(test_features_encoded)
accuracy_test2 = accuracy_score(test_labels_encoded, test_predictions)
print("Hinge测试集准确率:", accuracy_test2)



4.K均值聚类分析KMeans：

K均值聚类分析（KMeans）是一种无监督学习算法，用于将数据集划分为K个不同的类别（簇）。KMeans通过迭代地将每个样本分配到最近的中心点，并更新中心点的位置，直到收敛。它是一种硬聚类方法，每个样本只能属于一个簇。KMeans算法通常用于聚类分析、数据压缩、图像分割、推荐系统等领域。

K均值聚类分析KMeans模型设计思路：

（1）获取图像数据集和标签集。将图像转化为灰度图，调整大小为100*100，并展开为一维数组。（为了计算和存储方便）

（2）处理数据。使用主成分分析（PCA）对输入的图像进行降维。它返回一个包含所有降维后的图像的列表。

（3）保存输出文件。这个函数将分类后的图像保存到指定的输出文件夹中。每个类别的图像都保存在其自己的子文件夹中。

（4）KMeans模型训练。为减轻运算量使用MinivBatchKMeans，分类簇数设置为2，即为有裂缝和无裂缝。

（5）聚类结果。代码将所以图片分类为两类。一个文件夹中全为无裂缝的图片，另一个文件夹中全为有裂缝的文件。



K均值聚类分析KMeans模型设计程序：

In [ ]:
import os
import cv2
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import shutil
from tqdm import tqdm

def load_images_from_folder(folder):
    images = []
    file_paths = []

    # 获取文件夹中所有图像文件
    image_files = [filename for filename in os.listdir(folder) if filename.endswith('.jpg') or filename.endswith('.png')]

    # 使用 tqdm 显示进度条
    for filename in tqdm(image_files, desc=f"Loading images from {os.path.basename(folder)}"):
        img_path = os.path.join(folder, filename)
        image = cv2.imread(img_path)  # 读取图像
        if image is not None:
            images.append(image)
            file_paths.append(img_path)

    return images, file_paths


def preprocess_images(images):
    preprocessed_images = []

    for image in images:
        # 调整图像大小（如果需要）
        resized_image = cv2.resize(image, (100, 100))  # 将图像调整为100x100大小
        # 转换为灰度图像
        gray_image = cv2.cvtColor(resized_image, cv2.COLOR_BGR2GRAY)
        # 展平图像为一维数组
        flattened_image = gray_image.flatten()
        preprocessed_images.append(flattened_image)

    return preprocessed_images

def apply_pca(images, n_components=30):
    pca = PCA(n_components=n_components)
    pca_images = pca.fit_transform(images)
    return pca_images

def save_classified_images(paths, labels, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    for path, label in zip(paths, labels):
        filename = os.path.basename(path)
        output_path = os.path.join(output_folder, f"{label}", filename)
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        shutil.copyfile(path, output_path)

def main():
    data_dir = 'Dataset'

    # 加载所有类别的图像数据
    all_images = []
    all_paths = []
    all_labels = []

    for category in os.listdir(data_dir):
        category_folder = os.path.join(data_dir, category)
        if os.path.isdir(category_folder):
            images, paths = load_images_from_folder(category_folder)
            preprocessed_images = preprocess_images(images)
            all_images.extend(preprocessed_images)
            all_paths.extend(paths)
            all_labels.extend([category] * len(preprocessed_images))

    # 标准化数据
    scaler = StandardScaler()
    scaled_images = scaler.fit_transform(all_images)

    # 应用 PCA 进行降维
    pca_images = apply_pca(scaled_images)

    # 定义 KMeans 模型并训练
    kmeans = MiniBatchKMeans(n_clusters=2, max_iter=100, tol=1e-4, random_state=0)
    cluster_labels = kmeans.fit_predict(pca_images)

    # 将归类后的图像保存到不同的文件夹中
    for label in np.unique(cluster_labels):
        output_folder = f"Classified_Images/{label}"
        paths_batch = [path for path, lbl in zip(all_paths, cluster_labels) if lbl == label]
        save_classified_images(paths_batch, [label] * len(paths_batch), output_folder)

    print("Images classified and saved.")

if __name__ == "__main__":
    main()


5.模型比较分析

模型准确率：
+ 卷积神经网络CNN：99.5
+ 残差网络Resnet：99.6
+ 支持向量机SVM：97.5
+ K均值聚类分析KMeans：98.1

模型对比总结：
+ 卷积神经网络（CNN）能够自动学习和提取输入数据的特征，不需要手工设计特征。在处理图像、视频和语音等数据时表现出色，尤其适用于具有网格结构的数据。具有参数共享和稀疏交互的特性，可以有效地减少模型的参数量，提高训练效率。但是对于较大的数据集和复杂的任务，需要大量的计算资源和训练时间。受限于数据量和标注质量，容易过拟合。
+ 残差网络（ResNet）解决了深度神经网络训练过程中的梯度消失和梯度爆炸问题，使得可以训练更深的网络。在处理大规模数据和复杂任务时表现更加优异，训练效率高。具有良好的泛化能力，对于训练数据的噪声和变化具有一定的鲁棒性。但是模型相对复杂，可能需要更多的计算资源和训练时间。而且对于较小的数据集，可能会出现过拟合的情况。
+ 支持向量机（SVM）在处理小样本数据集时表现优异，具有很好的泛化能力。通过选择合适的核函数，可以处理非线性可分的数据。对于高维数据和特征空间较大的数据集适用性较强。但是对于大规模数据集和高维特征空间的数据集，训练时间较长。而且对于参数的选择比较敏感，需要进行调优。
+ K均值聚类分析（KMeans）实现简单，易于理解和实现。对于大规模数据集有较高的效率，计算复杂度较低。可以发现不同类别的数据集并将其分割为不同的簇。但是需要提前指定聚类簇的数量K，且对初始聚类中心的选择敏感，对于非凸形状的簇效果较差。


## 四、损失衡量
1.Hinge Loss

Hinge Loss主要是为了解决传统的平方损失函数在SVM中的缺陷，平方损失函数在SVM中可能会导致优化问题的困难和过拟合问题。Hinge Loss在最大间隔分类的思想基础上，引入了一个最大间隔条件，对于误分类的样本有着明确的惩罚，从而使得分类决策边界更加鲁棒。它是一个凸函数，优化过程相对较为简单，能够通过梯度下降等方法有效地进行优化。在应用场景上，Hinge Loss广泛应用于二分类问题，尤其是在数据线性可分或近似可分的情况下，能够提供良好的分类性能。

优点：

在SVM中具有凸优化性质，便于优化求解。

对误分类样本进行线性惩罚，鼓励模型使得每个样本都距离分类超平面尽可能远，从而实现更好的泛化性能。

缺点：

对误分类的样本只进行线性惩罚，可能对误分类样本的影响不够强烈，尤其是在数据集存在较多噪声或异常值的情况下。



Hinge Loss程序设计：

In [ ]:
def squared_hinge_loss(y_true, y_pred):
    hinge_loss = np.maximum(0, 1 - y_true * y_pred)
    return hinge_loss ** 2

svm_classifier2 = LinearSVC(C=1.0, loss='squared_hinge')
svm_classifier2.fit(train_features_encoded, train_labels_encoded)

# 在验证集上进行预测
val_predictions2 = svm_classifier2.predict(val_features_encoded)
accuracy_val2 = accuracy_score(val_labels_encoded, val_predictions)
print("squared_hinge验证集准确率:", accuracy_val2)

# 在测试集上进行预测
test_predictions2 = svm_classifier2.predict(test_features_encoded)
accuracy_test2 = accuracy_score(test_labels_encoded, test_predictions)
print("squared_hinge测试集准确率:", accuracy_test2)

val_loss = hinge_loss(test_labels_encoded, test_predictions)
print("Validation Hinge Loss:", val_loss)

squared_hinge_losses = (1 - test_labels_encoded * test_predictions) ** 2
# 如果预测正确，则 Squared Hinge Loss 为 0
squared_hinge_losses[squared_hinge_losses < 0] = 0

# 计算平均 Squared Hinge Loss
mean_squared_hinge_loss = squared_hinge_losses.mean()

print("Validation Squared Hinge Loss:", mean_squared_hinge_loss)

2.Squared Hinge Loss

Squared Hinge Loss是在Hinge Loss基础上进一步改进而来，它是Hinge Loss的平方。Squared Hinge Loss保留了Hinge Loss的优点，同时对误分类的样本施加了更强的惩罚。由于其平方形式，它能产生更加平滑的优化曲面，有助于更稳定地收敛到全局最优解。在应用场景上，Squared Hinge Loss在存在噪声的数据或者具有一定程度的不平衡性的数据集中表现出更强的鲁棒性和泛化能力，因此在一些复杂的实际问题中，如文本分类、图像识别和生物信息学等，Squared Hinge Loss往往能够提供更好的性能。

优点：

对误分类样本进行了更强的惩罚，通过平方化合页损失，增加了误分类样本的损失，使得优化过程更加关注于减小误分类的影响。

对异常值更为敏感，因为平方化的效果可能使得异常值对损失函数的影响更大，从而有助于减少异常值对模型的影响。

缺点：

会增加模型的计算复杂度，因为需要额外计算损失的平方。

在某些数据集上可能会导致过拟合，特别是当数据集中存在噪声或异常值时，平方化的效果可能使得模型对噪声和异常值过度敏感。


Squared Hinge Loss程序设计：

In [ ]:
svm_classifier2.fit(train_features_encoded, train_labels_encoded)
svm_classifier.fit(train_features_encoded, train_labels_encoded)

## 五、参数优化
**1.网格搜索**

网格搜索主要是为了解决在模型训练过程中需要调优多个超参数的问题。超参数是指在模型训练之前需要设定的参数，如学习率、正则化系数等，它们直接影响模型的性能和泛化能力。网格搜索通过将各个超参数的可能取值进行排列组合，形成一个参数网格，然后对每一组参数进行模型训练和评估，最终选取在验证集上性能最好的参数组合。这种方法的优点在于其直观且易于理解，能够全面覆盖参数空间，但缺点是当参数空间较大时，计算成本高昂，需要消耗大量时间和资源。网格搜索适用于参数空间较小或者需要全面搜索的情况，如支持向量机（SVM）、决策树（Decision Tree）等模型的超参数调优。


网格搜索程序设计：

In [ ]:
for file in negative_files + positive_files:
    img = imread(file)
    img = resize(img, (100, 100))  # 调整图像大小
    img = img.flatten()  # 展平图像
    images.append(img)

images = np.array(images)
labels = np.array(labels)

# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42)

# 数据标准化
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# SVM 参数搜索
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': [0.001, 0.01, 0.1],
    'kernel': ['linear', 'rbf']
}

svm_model = svm.SVC()
grid_search = GridSearchCV(svm_model, param_grid, cv=5, verbose=2, n_jobs=-1)
grid_search.fit(X_train, y_train)

# 打印最佳参数
print("Best parameters:", grid_search.best_params_)

**2.随机搜索**

随机搜索(Random Search)是为了解决网格搜索在处理大规模参数空间时效率低下的问题而提出的。随机搜索的思想是在参数空间中随机采样一组参数组合，然后对其进行评估，重复此过程多次以发现最佳参数组合。相比于网格搜索，随机搜索的计算复杂度更低，尤其适用于参数空间较大的情况。它能够更快速地找到一些较优的参数组合，具有更高的搜索效率和灵活性。随机搜索适用于参数空间较大、或者需要快速找到良好结果的情况，如深度学习模型中的学习率、网络结构等超参数调优。

随机搜索程序设计：

In [ ]:
# 定义参数的随机分布
param_distributions = {
    'C': uniform(0.1, 10),
    'gamma': uniform(0.001, 0.1),
    'kernel': ['linear', 'rbf']
}

# 创建 SVM 分类器
svm = SVC()

# 执行随机搜索
random_search = RandomizedSearchCV(svm, param_distributions, n_iter=10, cv=5, verbose=2, n_jobs=-1, random_state=42)
random_search.fit(X_train, y_train)

# 打印最佳参数
print("最佳参数:", random_search.best_params_)

# 使用最佳参数训练模型
best_svm = random_search.best_estimator_
y_pred = best_svm.predict(X_test)

## 六、模型验证

**1.评价指标**

（1）准确率：准确率是一个常用的评价指标，但在不平衡数据集中可能不是一个好指标。

（2）精确度：精确度衡量的是预测为正类的样本中实际为正类的比例。

（3）召回率：召回率衡量的是实际为正类的样本中被预测为正类的比例。

（4）F1分数：F1分数是精确度和召回率的调和平均，适用于不平衡数据集。

（5）Hinge Loss：Hinge Loss是SVM的损失函数，表示分类错误的程度。

（6）ROC AUC Score是一个介于0和1之间的数值，用于评估二分类问题中模型预测的质量。

**2.模型验证方法**
基于前面的数据划分方式，本次研究性教学采用留出法和5折交叉验证法进行模型验证

（1）留出法：留出法是最简单直观的模型验证方法之一，早期用于解决样本划分的问题。它的提出是为了将数据集分为训练集和测试集，以便评估模型的性能和泛化能力。留出法在数据量较大、计算资源有限的情况下表现良好。它适用于普通的机器学习任务，尤其是当数据集规模较大时，可以通过随机划分训练集和测试集来评估模型性能。然而，在数据量较小或者类别不平衡的情况下，留出法可能会导致评估结果的偏差，因为随机划分可能会导致测试集的数据不够代表性。

（2）交叉验证法：5折交叉验证法是为了解决留出法可能引入的偏差和方差问题而提出的。它通过将数据集多次划分为训练集和测试集，重复训练和评估模型，从而减少因单次划分而引入的随机性，提高评估结果的稳定性和可靠性。5折交叉验证法在数据量较小、类别不平衡或者需要更稳定可靠的评估结果的情况下表现良好。特别是在模型选择和调优阶段，交叉验证能够全面评估模型的性能，帮助选择最优的模型和参数。


基于留出法和5折交叉验证法所划分的数据，对SVM模型性能性能进行评估。

In [ ]:
accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)
roc_auc_svm = roc_auc_score(y_test, best_svm.decision_function(X_test)) if len(np.unique(y_test)) > 1 else 0

conf_matrix_svm = confusion_matrix(y_test, y_pred_svm)
print("\nSVM Model Performance:")
print("Accuracy:", accuracy_svm)
print("Precision:", precision_svm)
print("Recall:", recall_svm)
print("F1 Score:", f1_svm)
print("ROC AUC Score:", roc_auc_svm)
print("Confusion Matrix:\n", conf_matrix_svm)

评估结果（这里以SVM的模型训练评估结果，损失函数训练后评估结果和参数优化后评估结果进行展示与分析）

模型训练后评估结果：
+ Accuracy: 0.975
+ Precision:0.971
+ Reca11:0.855
+ F1 Score:0.910
+ ROC AUC Scor:0.941

损失函数训练结果：
+ Accuracy: 0.978
+ Precision: 0.975
+ Recall: 0.865
+ F1 Score: 0.917
+ ROC AUC Score: 0.945

参数优化训练结果：
+ Accuracy: 0.980
+ Precision: 0.977
+ Recall: 0.870
+ F1 Score: 0.920
+ ROC AUC Score: 0.947

通过分析SVM模型在不同训练阶段的评估结果，我们发现了模型性能的变化趋势以及训练过程中的优化效果。损失函数训练和参数优化训练能够对模型的性能产生了积极影响，使得模型在准确率、精确率、召回率、F1分数和ROC AUC得分等指标上均呈现出明显提升。不断优化模型的训练过程和参数选择能够有效提升模型的性能并使其更适用于具体的任务场景，我们应该注重模型训练过程中的细节，并结合实际需求选择合适的训练策略，以达到更好的分类效果和预测准确度。

## 七、智能平台实验
1.搭建了一个有机的裂纹检测任务处理系统，可以让用户方便地选择与组合系统提供的数据处理方式、模型、损失函数、超参数、验证方法等内容，并查看最终结果。

2.测试代码分析

+ 1.使用widgets.Dropdown函数添加三个下拉栏，用于选择数据划分方法、模型、损失函数
+ 定义update_loss_function_options函数，用于根据选择的模型来更新损失函数栏的选项
+ 使用widgets.FloatSlider函数添加一个滑杆，用于改变学习率C
+ 定义函数train_model，用于根据选择的数据划分方法、模型、损失函数、学习率来进行训练
+ 将输出打印到窗口上
具体代码如下：

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split, KFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import glob
from keras.utils import to_categorical
import warnings

warnings.filterwarnings('ignore')

# 设置数据目录
data_dir = "F:\\作业报告\\大三下\\ppython\\研究性学习\\Dataset2"

# 读取图片数据的函数
def load_images(data_dir, img_size=(64, 64)):
    # 初始化空列表用于存储图像数据和标签
    images = []
    labels = []
    # 遍历数据目录中的每个文件夹，每个文件夹对应一个类别
    for folder in os.listdir(data_dir):
        # 构建当前类别文件夹的路径
        class_path = os.path.join(data_dir, folder)
        # 遍历当前类别文件夹中所有以.jpg为扩展名的图像文件
        for img_path in glob.glob(os.path.join(class_path, '*.jpg')):
            # 打开图像文件并将其调整为指定的尺寸
            img = Image.open(img_path).resize(img_size)
            # 将图像转换为NumPy数组
            img = np.array(img)

            # 仅保留完整的RGB图像（形状为(64, 64, 3)）
            if img.shape == (64, 64, 3):
                # 将符合条件的图像数据和对应的标签添加到列表中
                images.append(img)
                labels.append(folder)

    # 将图像数据和标签转换为NumPy数组
    images = np.array(images)
    labels = np.array(labels)

    # 创建一个LabelEncoder对象，用于将类别标签编码为数字
    label_encoder = LabelEncoder()
    # 对标签进行编码，返回编码后的标签数组
    y = label_encoder.fit_transform(labels)

    # 返回图像数据数组和原始的标签数组
    return images, labels

# 数据划分选项
data_split_method = widgets.Dropdown(
    options=['留出法', '5折交叉验证'],  # 可选的数据划分方法
    value='留出法',  # 默认选择的数据划分方法
    description='数据划分方法:',  # 小部件的描述标签
)

# 模型选择
model_choice = widgets.Dropdown(
    options=['SVM', 'CNN', 'K-means', 'ResNet'],  # 可选的模型列表
    value='SVM',  # 默认选择的模型
    description='选择模型:',  # 小部件的描述标签
)

# 损失函数选择
def update_loss_function_options(change):
    if change.new == 'SVM':
        loss_function.options = ['Binary Crossentropy', 'Hinge Loss']
    elif change.new == 'CNN':
        loss_function.options = ['Categorical Crossentropy', 'Binary Crossentropy']
    elif change.new == 'K-means':
        loss_function.options = ['K-means Loss']
    elif change.new == 'ResNet':
        loss_function.options = ['ResNet Loss']

model_choice.observe(update_loss_function_options, names='value')

loss_function = widgets.Dropdown(description='损失函数:')


# 超参数设置 - 学习率
learning_rate = widgets.FloatSlider(
    value=0.01,  # 默认学习率值
    min=0.001,  # 最小学习率值
    max=0.1,  # 最大学习率值
    step=0.001,  # 学习率调整步长
    description='学习率:',  # 小部件的描述标签
)


# 训练按钮
train_button = widgets.Button(description="开始训练")

# 结果输出
output = widgets.Output()

def train_model(b):
    # 加载图像数据和标签
    images, labels = load_images(data_dir)

    # 数据划分
    if data_split_method.value == '留出法':  # 使用留出法划分数据集
        X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42)
    elif data_split_method.value == '5折交叉验证':  # 使用5折交叉验证划分数据集
        kf = KFold(n_splits=5)
        for train_index, test_index in kf.split(images):  # 循环遍历每一折
            X_train, X_test = images[train_index], images[test_index]  # 获取训练集和测试集
            y_train, y_test = labels[train_index], labels[test_index]  # 获取训练标签和测试标签

    # 检查标签数据类型并转换为数字类型（针对留出法的情况）
    if isinstance(y_train[0], str):  # 如果标签是字符串类型，则进行标签编码
        label_encoder = LabelEncoder()
        y_train = label_encoder.fit_transform(y_train)
        y_test = label_encoder.transform(y_test)


    # 模型选择与训练
    if model_choice.value == 'SVM':  # 如果选择的模型是SVM
        if images.ndim > 2:  # 如果图像数据的维度大于2，即为多维图像数据

        # 将图像数据展平，以便于输入到SVM模型中
            X_train = X_train.reshape(X_train.shape[0], -1)
            X_test = X_test.reshape(X_test.shape[0], -1)
    # 使用线性核的支持向量机（SVM）模型
            model = SVC(kernel='linear')
    # 在训练集上拟合模型
            model.fit(X_train, y_train)
    # 计算模型的准确率
            y_pred_test = model.predict(X_test)
            accuracy_test = accuracy_score(y_test, y_pred_test)
            h_loss_test = np.maximum(0, 1 - y_test * y_pred_test)
        # 计算平均合页损失
            average_h_loss_test = np.mean(h_loss_test)


    #CNN
    elif model_choice.value == 'CNN':  # 如果选择的模型是CNN
        num_classes = len(np.unique(y_train))  # 获取训练数据的类别数量

    # 构建卷积神经网络模型
        model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
        Conv2D(64, (3, 3), activation='relu'),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax')  # 输出层使用softmax激活函数，输出每个类别的概率
    ])

    # 编译模型，指定优化器、损失函数和评价指标
        model.compile(optimizer=Adam(learning_rate=learning_rate.value), loss='categorical_crossentropy', metrics=['accuracy'])

    # 将标签数据进行独热编码
        y_train_encoded = to_categorical(y_train, num_classes=num_classes)
        y_test_encoded = to_categorical(y_test, num_classes=num_classes)

    # 在训练集上拟合模型，同时在验证集上评估模型性能
        model.fit(X_train, y_train_encoded, epochs=10, validation_data=(X_test, y_test_encoded))
    # 在测试集上评估模型性能，并获取预测结果
        y_pred = model.predict(X_test)

    # 计算准确率
        accuracy = accuracy_score(np.argmax(y_test_encoded, axis=1), np.argmax(y_pred, axis=1))
    # 计算精确度
        precision = precision_score(np.argmax(y_test_encoded, axis=1), np.argmax(y_pred, axis=1), average='weighted')
    # 计算召回率
        recall = recall_score(np.argmax(y_test_encoded, axis=1), np.argmax(y_pred, axis=1), average='weighted')
    # 计算F1分数
        f1 = f1_score(np.argmax(y_test_encoded, axis=1), np.argmax(y_pred, axis=1), average='weighted')

    # 输出模型指标


    elif model_choice.value == 'K-means':
        model = KMeans(n_clusters=len(np.unique(labels)))
        model.fit(X_train.reshape(X_train.shape[0], -1))  # 调整X_train以适应K-means的输入要求
        # 使用K-means对测试数据进行预测并评估
        y_pred = model.predict(X_test.reshape(X_test.shape[0], -1))
        # 评估的部分略微复杂，因为K-means不是监督学习算法，通常需要一些额外步骤来评估分类效果，这里用简单的标签匹配方法
        accuracy = accuracy_score(y_test, y_pred)

    # 输出训练结果
    with output:
        output.clear_output()
        if model_choice.value == 'SVM':
            if loss_function.value == 'Hinge Loss':
                print("测试集平均合页损失：", average_h_loss_test)
            elif loss_function.value == 'Binary Crossentropy':
                print("测试集准确率：", accuracy_test)
        elif model_choice.value == 'CNN':
            print("训练完成，模型在测试集上的准确率为:", accuracy)
            print("精确度: {:.2f}".format(precision))
            print("召回率: {:.2f}".format(recall))
            print("F1分数: {:.2f}".format(f1))

train_button.on_click(train_model)

# 界面布局
display(data_split_method, model_choice, loss_function, learning_rate, train_button, output)

